In [24]:
# REQUIRES
# ======================================================================================================================================
import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import pandas as pd
import polars as pl


In [ ]:
# GLOBALS
# ======================================================================================================================================
PATH_MOVIES  = "../dataset/ml-20m/movies.csv"
PATH_RATINGS = "../dataset/ml-20m/ratings.csv"

spark = SparkSession.builder \
  .appName("AnalysisDataset") \
  .master("local[*]") \
  .config("spark.driver.memory", "64g") \
  .getOrCreate()

if spark.sparkContext._jsc.sc().isStopped():
  print("Spark session is not running.")
  sys.exit()

In [ ]:
# HELPERS FUNCTIONS
# ======================================================================================================================================

def LineBreak(pHowMany: int = 1):
  for i in range(0, pHowMany):
    print(f"+ ")

def SectionBreak():
  LineBreak(1)
  print(f"+ --------------------------------------------------------------------------")
  LineBreak(1)



In [ ]:
# 1. Cargar el dataset movielens superdataset.csv (ratings.csv in PATH_RATINGS)
# ======================================================================================================================================

df_pd = {}
df_pl = {}
df_sp = {}

print(f"+ =======================================================================")
print(f"+ Load '{PATH_RATINGS}' from the movielens dataset ")
print(f"+ -----------------------------------------------------------------------")
LineBreak(1)
print(f"+ Loading with Pandas...")

tInit = time.time()
df_pd['ratings'] = pd.read_csv(PATH_RATINGS)
tEnd = time.time()
df_pd['movies'] = pd.read_csv(PATH_MOVIES)

print(f"+ Time to load the dataset with Pandas: {tEnd - tInit:.4f} seconds")
SectionBreak()
print(f"+ Loading with Polars...")

tInit = time.time()
df_pl['ratings'] = pl.read_csv(PATH_RATINGS)
tEnd = time.time()
df_pl['movies'] = pl.read_csv(PATH_MOVIES)

print(f"+ Time to load the dataset with Polars: {tEnd - tInit:.4f} seconds")
SectionBreak()
print(f"+ Loading with PySpark...")

tInit = time.time()
df_sp['ratings'] = spark.read.csv(PATH_RATINGS, header=True, inferSchema=True)
countSp = df_sp['ratings'].count()
tEnd = time.time()
df_sp['movies'] = spark.read.csv(PATH_MOVIES, header=True, inferSchema=True)
# Only if we want to force the keep dataset in cache
# df_sp.cache()

print(f"+ Time to load the dataset with PySpark: {tEnd - tInit:.4f} seconds for {countSp} rows")
LineBreak(1)
print(f"+ =======================================================================")

+ =======================================================================
+ Load '../dataset/ml-20m/ratings.csv' from the movielens dataset 
+ -----------------------------------------------------------------------
+ 
+ Loading with Pandas...
+ Time to load the dataset with Pandas: 7.9170 seconds
+ 
+ -----------------------------------------------------------------------
+ 
+ Loading with Polars...
+ Time to load the dataset with Polars: 0.1830 seconds
+ 
+ -----------------------------------------------------------------------
+ 
+ Loading with PySpark...


+ Time to load the dataset with PySpark: 2.6293 seconds for 20000263 rows
+ 
+ =======================================================================


In [ ]:
# 2. Filtrar películas de género "Action" con rating ≥ 4
# ======================================================================================================================================

def GenreFilterAction(pBy: str = "pandas", pGnr: str = "Action", pRat: int = 4):
  print(f"+ -----------------------------------------------------------------------")
  LineBreak(1)
  print(f"+ Filtering with {pBy.upper()} by {pGnr.upper()} and rating >= {pRat} ...")

  if "pandas" in pBy.lower():
    GenreFilterActionByPandas(pGnr, pRat)
    pass
  if "polars" in pBy.lower():
    GenreFilterActionByPolars(pGnr, pRat)
    pass
  if "pyspark" in pBy.lower():
    GenreFilterActionByPySpark(pGnr, pRat)

def GenreFilterActionByPandas(pGnr: str = "Action", pRat: int = 4):
  tInit = time.time()
  avgRatingsPd = (df_pd['ratings']
    .groupby('movieId')['rating']
    .mean()
    .reset_index()
  )
  dfJoinedPd = avgRatingsPd.merge(df_pd['movies'], on='movieId')
  bFilterPd = (dfJoinedPd['genres'].str.contains(pGnr)) & (dfJoinedPd['rating'] >= pRat)
  resPd = dfJoinedPd[bFilterPd]
  print(f"+ Pandas result: {len(resPd)} Filtered at Time: {time.time() - tInit:.4f}s")
  LineBreak(1)

def GenreFilterActionByPolars(pGnr: str = "Action", pRat: int = 4):
  tInit = time.time()
  resPl = (df_pl['ratings']
    .group_by("movieId")
    .agg(pl.col("rating").mean().alias("avgRating"))
    .join(df_pl['movies'], on="movieId")
    .filter((pl.col("genres").str.contains(pGnr)) & (pl.col("avgRating") >= pRat))
  )
  print(f"+ Polars result: {resPl.height} Filter Time: {time.time() - tInit:.4f}s")
  LineBreak(1)

def GenreFilterActionByPySpark(pGnr: str = "Action", pRat: int = 4):
  tInit = time.time()
  resSp = (df_sp['ratings']
    .groupBy("movieId")
    .agg(F.avg("rating").alias("avgRating"))
    .join(df_sp['movies'], "movieId")
    .filter((F.col("genres").contains(pGnr)) & (F.col("avgRating") >= pRat))
  )
  countSp = resSp.count()
  tEnd = time.time()
  print(f"+ PySpark result: {countSp}  Filter Time: {tEnd - tInit:.4f}s")
  LineBreak(1)
  

print(f"+ =======================================================================")
print(f"+ Fillter movies by GENRE with rating ≥ N ...")

GenreFilterAction("pandas",  "Action", 4)
GenreFilterAction("polars",  "Action", 4)
GenreFilterAction("pyspark", "Action", 4)

print(f"+ =======================================================================")


+ =======================================================================
+ Filtrar películas de género 'Action' con rating ≥ 4
+ -----------------------------------------------------------------------
+ 
+ Filtering with PANDAS by ACTION and rating >= 4 ...
+ Pandas result: 118 Filtered at Time: 1.1673s
+ 
+ -----------------------------------------------------------------------
+ 
+ Filtering with POLARS by ACTION and rating >= 4 ...
+ Polars result: 118 Filter Time: 0.3525s
+ 
+ -----------------------------------------------------------------------
+ 
+ Filtering with PYSPARK by ACTION and rating >= 4 ...
+ PySpark result: 118  Filter Time: 0.2666s
+ 
+ =======================================================================


In [ ]:
# 3. Calcular media y desviación estándar de ratings por usuario
# ======================================================================================================================================

def UserStats():
  print(f"+ ==========================================================================")
  print(f"+ Caltulating user stats (ratings average and standard deviation by user)...")
  print(f"+ --------------------------------------------------------------------------")
  LineBreak(1)
  
  tInit = time.time()
  userStatsPd = df_pd['ratings'].groupby('userId')['rating'].agg(['mean', 'std'])
  print(f"+ Pandas Stats({len(userStatsPd)}) Time: {time.time() - tInit:.4f}s")

  SectionBreak()
  
  tInit = time.time()
  userStatsPl = df_pl['ratings'].group_by("userId").agg([
    pl.col("rating").mean().alias("mean"),
    pl.col("rating").std().alias("std")
  ])
  print(f"+ Polars Stats({userStatsPl.height}) Time: {time.time() - tInit:.4f}s")

  SectionBreak()
  
  tInit = time.time()
  userStatsSp = df_sp['ratings'].groupBy("userId").agg(
    F.avg("rating").alias("mean"),
    F.stddev("rating").alias("std")
  )
  print(f"+ PySpark Stats({userStatsSp.cache().count()}) Time: {time.time() - tInit:.4f}s")
  
  LineBreak(1)
  print(f"+ ==========================================================================")
  
UserStats()

+ ==========================================================================
+ Caltulating user stats (ratings average and standard deviation by user)...
+ --------------------------------------------------------------------------
+ 
+ Pandas Stats(138493 ) Time: 0.9745s
+ 
+ --------------------------------------------------------------------------
+ 
+ Polars Stats(138493) Time: 0.2680s
+ 
+ --------------------------------------------------------------------------
+ 


+ PySpark Stats(138493) Time: 2.8422s
+ 
+ ==========================================================================


In [ ]:
# 4. Ordenar películas por rating promedio
# ======================================================================================================================================

def SortMoviesByRating():
  print(f"+ ==========================================================================")
  print(f"+ Sorting movies by average rating...")
  print(f"+ --------------------------------------------------------------------------")
  LineBreak(1)
  
  tInit = time.time()
  resSortedPd = df_pd['ratings'].groupby('movieId')['rating'].mean().sort_values(ascending=False)
  print(f"+ Pandas Sort({len(resSortedPd)}) Time: {time.time() - tInit:.4f}s")

  SectionBreak()

  tInit = time.time()
  resSortedPl = (df_pl['ratings']
    .group_by("movieId")
    .agg(pl.col("rating").mean())
    .sort("rating", descending=True)
  )
  print(f"+ Polars Sort({resSortedPl.height}) Time: {time.time() - tInit:.4f}s")

  SectionBreak()

  tInit = time.time()
  resSortedSp = (df_sp['ratings']
    .groupBy("movieId")
    .agg(F.avg("rating").alias("avgRating"))
    .orderBy(F.desc("avgRating"))
  )
  countSp = resSortedSp.count()
  tEnd = time.time()
  resSortedSp.show(5)
  print(f"+ PySpark Sort({countSp}) Time: {tEnd - tInit:.4f}s")

  LineBreak(1)
  print(f"+ ==========================================================================")

SortMoviesByRating()

+ ==========================================================================
+ Sorting movies by average rating...
+ --------------------------------------------------------------------------
+ 
+ Pandas Sort(26744) Time: 1.1072s
+ 
+ --------------------------------------------------------------------------
+ 
+ Polars Sort(26744) Time: 0.3161s
+ 
+ --------------------------------------------------------------------------
+ 


+-------+---------+
|movieId|avgRating|
+-------+---------+
| 100743|      5.0|
| 129293|      5.0|
| 129741|      5.0|
| 103912|      5.0|
| 112790|      5.0|
+-------+---------+
only showing top 5 rows


+ PySpark Sort(26744) Time: 6.2976s


In [ ]:
# 5. Convertir rating timestamp a fecha
# ======================================================================================================================================

def Ts2DateConversion():
  print(f"+ ==========================================================================")
  print(f"+ Preparing the dataframes (preparation & date conversion)...")
  print(f"+ --------------------------------------------------------------------------")
  LineBreak(1)
  
  print(f"+ With Pandas...")
  tInit = time.time()
  df_pd['ratings']['date'] = pd.to_datetime(df_pd['ratings']['timestamp'], unit='s')
  print(f"+ Pandas Date Conversion: {time.time() - tInit:.4f}s")
  
  SectionBreak()
  
  print(f"+ With Polars...")
  tInit = time.time()
  df_pl['ratings'] = df_pl['ratings'].with_columns(pl.from_epoch("timestamp", time_unit="s").alias("date"))
  print(f"+ Polars Date Conversion: {time.time() - tInit:.4f}s")
  
  SectionBreak()
  
  print(f"+ With PySpark...")
  tInit = time.time()
  df_sp['ratings'] = df_sp['ratings'].withColumn("date", F.from_unixtime("timestamp").cast("date"))
  print(f"+ PySpark Date Conversion: {time.time() - tInit:.4f}s")
  
  LineBreak(1)
  print(f"+ ==========================================================================")
  
Ts2DateConversion()

+ ==========================================================================
+ Preparing the dataframes (preparation & date conversion)...
+ --------------------------------------------------------------------------
+ 
+ With Pandas...
+ Pandas Date Conversion: 15.0056s
+ 
+ --------------------------------------------------------------------------
+ 
+ With Polars...
+ Polars Date Conversion: 0.0429s
+ 
+ --------------------------------------------------------------------------
+ 
+ With PySpark...
+ PySpark Date Conversion: 0.0160s
+ 
+ ==========================================================================
